In [1]:
import os
import sys
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
import torchvision
from lightning.pytorch.loggers import MLFlowLogger
from sklearn.utils.class_weight import compute_class_weight
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchmetrics import F1Score
from torchmetrics.classification import Accuracy, ConfusionMatrix
from torchvision import models

In [2]:
SEED = 42
torch.manual_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [3]:
BASE_DIR = Path(os.getcwd()).parent
sys.path.append(str(BASE_DIR))

from api.app.v1.flowers.models import FlowerDataset
from api.app.v1.flowers.training_utils import get_device
from src.data import FlowerDataModule

In [4]:
device = get_device()
data_dir = BASE_DIR / "data"
model_chx_dir = BASE_DIR / "models_checkpoints"
os.makedirs(model_chx_dir, exist_ok=True)

Using device: cuda


In [5]:
pretrained_model = models.efficientnet_b0(weights="IMAGENET1K_V1")

In [6]:
print(help(pretrained_model.features[-1]))

Help on Conv2dNormActivation in module torchvision.ops.misc object:

class Conv2dNormActivation(ConvNormActivation)
 |  Conv2dNormActivation(
 |      in_channels: int,
 |      out_channels: int,
 |      kernel_size: Union[int, tuple[int, int]] = 3,
 |      stride: Union[int, tuple[int, int]] = 1,
 |      padding: Union[int, tuple[int, int], str, NoneType] = None,
 |      groups: int = 1,
 |      norm_layer: Optional[Callable[..., torch.nn.modules.module.Module]] = <class 'torch.nn.modules.batchnorm.BatchNorm2d'>,
 |      activation_layer: Optional[Callable[..., torch.nn.modules.module.Module]] = <class 'torch.nn.modules.activation.ReLU'>,
 |      dilation: Union[int, tuple[int, int]] = 1,
 |      inplace: Optional[bool] = True,
 |      bias: Optional[bool] = None
 |  ) -> None
 |
 |  Configurable block used for Convolution2d-Normalization-Activation blocks.
 |
 |  Args:
 |      in_channels (int): Number of channels in the input image
 |      out_channels (int): Number of channels produ

In [7]:
print(pretrained_model.features[-1].out_channels)

1280


In [8]:
print(pretrained_model.classifier[1])

Linear(in_features=1280, out_features=1000, bias=True)


In [9]:
print(type(pretrained_model))

<class 'torchvision.models.efficientnet.EfficientNet'>


In [10]:
print(pretrained_model.classifier.parameters)
print(pretrained_model.classifier[1].in_features)

<bound method Module.parameters of Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)>
1280


In [11]:
labels = FlowerDataset(data_dir).labels
class_weights_np = compute_class_weight(
    class_weight="balanced", classes=np.unique(labels), y=labels
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

Dataset not found at /home/zelluzy/Desktop/code/flowers/data/flowers-102.                 Downloading using torchvision...


100%|██████████| 345M/345M [00:19<00:00, 17.5MB/s] 
100%|██████████| 502/502 [00:00<00:00, 3.79MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 72.3MB/s]


In [12]:
from collections.abc import Mapping
from typing import Any


class FlowerClassifier(L.LightningModule):
    def __init__(
        self,
        pretrained_model: nn.Module,
        num_classes: int,
        class_weights: torch.Tensor | None,
        lr_head: float = 1e-3,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(
            "lr_head", "num_classes", ignore=["backbone", "class_weights"]
        )
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_classes),
        )
        self.model = pretrained_model
        # replace classifier
        self.model.classifier[1] = nn.Linear(  # type: ignore
            pretrained_model.classifier[1].in_features,  # type: ignore
            num_classes,
        )

        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)

        self.train_f1 = F1Score(
            task="multiclass", num_classes=num_classes, average="macro"
        )
        self.val_f1 = F1Score(
            task="multiclass", num_classes=num_classes, average="macro"
        )
        self.test_f1 = F1Score(
            task="multiclass", num_classes=num_classes, average="macro"
        )

        self.per_class_f1 = F1Score(
            task="multiclass", num_classes=num_classes, average=None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def training_step(
        self, batch, batch_idx
    ) -> torch.Tensor | Mapping[str, Any] | None:
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.train_acc(logits, y)
        self.train_f1(logits, y)
        self.log("train_loss", loss, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc, on_epoch=True, prog_bar=True)
        self.log("train_f1", self.train_f1, prog_bar=True)
        return loss

    def validation_step(
        self, batch, batch_idx
    ) -> torch.Tensor | Mapping[str, Any] | None:
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.val_acc(logits, y)
        self.val_f1(logits, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_acc, prog_bar=True)
        self.log("val_f1", self.val_f1, prog_bar=True)
        return loss

    def on_validation_epoch_end(self) -> None:
        return super().on_validation_epoch_end()

    def test_step(self, batch, batch_idx) -> torch.Tensor | Mapping[str, Any] | None:
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.test_acc(logits, y)
        self.test_f1(logits, y)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", self.test_acc, prog_bar=True)
        self.log("test_f1", self.test_f1, prog_bar=True)
        return loss

    def configure_optimizers(self) -> dict[str, Any]:
        # first only classifier params
        optimizer = optim.AdamW(
            self.model.classifier.parameters(), lr=self.hparams.lr_head
        )  # type: ignore

        assert isinstance(self.trainer.max_epochs, int)
        lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": lr_scheduler,
            "interval": "epoch",
            "monitor": "val_loss",
        }

In [13]:
from lightning import LightningModule
from lightning.pytorch.callbacks import BaseFinetuning
from torch.optim import Optimizer


class BackboneFinetuning(BaseFinetuning):
    def __init__(self, unfreeze_at_epoch: int = 5, lr_backbone: float = 1e-5) -> None:
        super().__init__()
        self.unfreeze_at_epoch = unfreeze_at_epoch
        self.lr_backbone = lr_backbone

    def freeze_before_training(self, pl_module: LightningModule) -> None:
        """freeze backbone for phase 1"""
        self.freeze(pl_module.model.features)  # type: ignore

    def finetune_function(
        self, pl_module: LightningModule, epoch: int, optimizer: Optimizer
    ) -> None:
        if epoch == self.unfreeze_at_epoch:
            self.unfreeze_and_add_param_group(
                modules=pl_module.model.features,  # type: ignore
                optimizer=optimizer,
                lr=self.lr_backbone,
            )

            # Patch the scheduler so it tracks the backbone
            scheduler = pl_module.lr_schedulers()
            scheduler.base_lrs.append(self.lr_backbone)  # type: ignore

In [14]:
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
    StochasticWeightAveraging,
)

trc_uri = f"sqlite:///{BASE_DIR}/mlflow.db"
exp_name = "flower-classification-v2"

# explicitly set artifact location for mlflow
mlflow.set_tracking_uri(trc_uri)
experiment = mlflow.get_experiment_by_name(exp_name)
if experiment is None:
    mlflow.create_experiment(exp_name, artifact_location=f"{BASE_DIR}/artifacts/")

mlflow_logger = MLFlowLogger(
    experiment_name=exp_name,
    tracking_uri=trc_uri,
    log_model=True,
)

callbacks = [
    BackboneFinetuning(unfreeze_at_epoch=5, lr_backbone=1e-5),
    ModelCheckpoint(
        monitor="val_acc",
        mode="max",
        save_top_k=1,
        filename="best-{epoch}-{val_acc:.3f}",
        dirpath=model_chx_dir,
    ),
    EarlyStopping(monitor="val_acc", mode="max", patience=5),
    # shows up as an MLflow metric — good for sanity-checking the unfreeze
    LearningRateMonitor(logging_interval="epoch"),
]

trainer = L.Trainer(
    max_epochs=30,
    logger=mlflow_logger,
    callbacks=callbacks,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    log_every_n_steps=10,
    accumulate_grad_batches=1,
)

flower_dm = FlowerDataModule(data_dir, batch_size=32)

pl_model = FlowerClassifier(
    pretrained_model=pretrained_model,
    num_classes=len(FlowerDataset(data_dir).classes),
    class_weights=class_weights,
)

mlflow_logger.log_hyperparams(
    {"effective_batch_size": flower_dm.batch_size * trainer.accumulate_grad_batches}
)

2026/07/12 12:50:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/12 12:50:43 INFO mlflow.store.db.utils: Updating database tables
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [15]:
print(pl_model.model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=102, bias=True)
)


In [16]:
trainer.fit(pl_model, datamodule=flower_dm)

You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/zelluzy/Desktop/code/flowers/models_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/core/optimizer.py:378: Found unsupported keys in the optimizer configuration: {'interval'}
/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary. 

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model        │ EfficientNet       │  4.1 M │ train │     0 │
│ 1 │ criterion    │ CrossEntropyLoss   │      0 │ train │     0 │
│ 2 │ train_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc      │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ test_acc     │ MulticlassAccuracy │      0 │ train │     0 │
│ 5 │ train_f1     │ MulticlassF1Score  │      0 │ train │     0 │
│ 6 │ val_f1       │ MulticlassF1Score  │      0 │ train │     0 │
│ 7 │ test_f1      │ MulticlassF1Score  │      0 │ train │     0 │
│ 8 │ per_class_f1 │ MulticlassF1Score  │      0 │ train │     0 │
└───┴──────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 172 K                                                                                            
Non-trainable params: 4.0 M                                                                                        
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16.553                                                                     
Modules in train mode: 345                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

`Trainer.fit` stopped: `max_epochs=30` reached.


In [17]:
checkpoint_callback = callbacks[1]  # ModelCheckpoint callback
best_model = FlowerClassifier.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    pretrained_model=pretrained_model,
    num_classes=102,
    class_weights=class_weights,
)

In [18]:
trainer.test(pl_model, datamodule=flower_dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9980812668800354     │
│          test_f1          │    0.9980666637420654     │
│         test_loss         │    0.02071443386375904    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.02071443386375904,
  'test_acc': 0.9980812668800354,
  'test_f1': 0.9980666637420654}]